In [1]:
# Setup: load artifacts from main pipeline
import pandas as pd
import textwrap
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

# Tuning parameters for evaluation
top_k_retrieval = 6  # increase if you want more context per question
max_new_tokens = 320
llm_temperature = 0.4
llm_top_p = 0.9

# Paths
CHUNKS_PKL = Path('index/byd_fgd1_chunks.pkl')
FAISS_INDEX_PATH = Path('index/byd_fgd1.index')
EMB_MODEL_PATH = Path('index/byd_fgd1_emb_model_name.txt')

# Load chunks + index + model
chunks_df = pd.read_pickle(CHUNKS_PKL)
emb_model_name = EMB_MODEL_PATH.read_text().strip()
retrieval_model = SentenceTransformer(emb_model_name)
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

def search_chunks(query, k=5):
    q_emb = retrieval_model.encode([query], normalize_embeddings=True)
    D, I = faiss_index.search(q_emb, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        row = chunks_df.iloc[idx]
        cid = row.get('chunk_id') or f"s{row['session_id']}_t{row['turn_start']}-{row['turn_end']}"
        results.append({
            'chunk_id': cid,
            'score': float(score),
            'session_id': str(row.get('session_id','')),
            'turn_start': int(row.get('turn_start',0)),
            'turn_end': int(row.get('turn_end',0)),
            'text': row.get('text',''),
        })
    return results


# Load local LLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
LOCAL_LLM_NAME = 'google/flan-t5-base'
_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)
_model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_LLM_NAME)
_gen = pipeline(
    "text2text-generation",
    model=_model,
    tokenizer=_tokenizer,
    max_new_tokens=max_new_tokens,
)
def build_prompt(context, question):
    return (
        "You are a research analyst reviewing a focus group transcript.\n"
        "Use only the context to answer the question.\n"
        "Answer in 2–3 sentences and cite specific details from the context.\n"
        "If the answer is not in the context, say 'Not enough context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )



Device set to use mps:0


In [2]:
# Define evaluation questions (edit this list or load from CSV)
questions = [
    "What were the main concerns about BYD pricing and value?",
    "What did participants say about charging convenience?",
    "What reasons were given for preferring other brands?",
    "How did participants describe BYD’s battery technology?",
    "What impressions did people have of the BYD interior and features?",
    "What did they think about ride comfort and driving experience?",
    "How important was resale value compared to other factors?",
    "What worries did they have about servicing or maintenance?",
    "How did they compare BYD to established car brands?",
    "What incentives or benefits were most appealing to them?",
    "How did they react to public charging discounts or programs?",
    "What feedback did they have on cargo space and seating?",
    "Did they express trust or skepticism about EV safety?",
    "What factors would persuade them to switch to BYD?",
    "How did home vs. public charging influence their decision?",
    "What did they say about design and styling?",
    "Which features were considered “must haves”?",
    "What budget or price thresholds were mentioned?",
    "How did long-distance travel affect their perception?",
    "What doubts remained after hearing the presentation?",
]

def run_rag(question, k=top_k_retrieval):
    hits = search_chunks(question, k=k)
    context = "\n\n".join(
        f"[{h["chunk_id"]}] {textwrap.shorten(h["text"], width=500, placeholder="...")}"
        for h in hits
    )
    prompt = build_prompt(context, question)
    answer = _gen(
        prompt,
        temperature=llm_temperature,
        top_p=llm_top_p,
    )[0]["generated_text"]
    return answer, hits

rows = []
for q in questions:
    ans, hits = run_rag(q)
    rows.append({
        "question": q,
        "answer": ans,
        "sources": ", ".join(h["chunk_id"] for h in hits),
        "context": "\n\n".join(h["text"] for h in hits),
    })

eval_df = pd.DataFrame(rows)
eval_df


Token indices sequence length is longer than the specified maximum sequence length for this model (1145 > 512). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,question,answer,sources,context
0,What were the main concerns about BYD pricing ...,"The marketing is very good, price wise also. I...",sBYD Brand Study_Transcript for G1_ICE Models ...,even my father knows about BYD more than me. S...
1,What did participants say about charging conve...,"It doesn't make a difference. For me, it doesn...",sBYD Brand Study_Transcript for G1_ICE Models ...,rator) [turn 94]: Nobody? Not even Angeline? O...
2,What reasons were given for preferring other b...,They are like McDonald's. They're everywhere. ...,sBYD Brand Study_Transcript for G1_ICE Models ...,"h BYD, ?\nAngeline (Respondent) [turn 478]: Ye..."
3,How did participants describe BYD’s battery te...,Not enough context.,sBYD Brand Study_Transcript for G1_ICE Models ...,think it has the other like BYD and Telsa. Tel...
4,What impressions did people have of the BYD in...,Luxurious and looks very luxurious,sBYD Brand Study_Transcript for G1_ICE Models ...,ou remember besides the model?\nSiti (Responde...
5,What did they think about ride comfort and dri...,"Comfort the features and then the dashboard, t...",sBYD Brand Study_Transcript for G1_ICE Models ...,"h? good. Gary, tell us about the positive impr..."
6,How important was resale value compared to oth...,Not enough context.,sBYD Brand Study_Transcript for G1_ICE Models ...,won't dive in on the left hand because of this...
7,What worries did they have about servicing or ...,Not enough context.,sBYD Brand Study_Transcript for G1_ICE Models ...,"etcetera. So, and 200 km is quite a lot lah I ..."
8,How did they compare BYD to established car br...,It holds quite a high value in terms of most o...,sBYD Brand Study_Transcript for G1_ICE Models ...,oo common already. I won't buy a BYD. I'll buy...
9,What incentives or benefits were most appealin...,"The food and beverages, the charging discounts...",sBYD Brand Study_Transcript for G1_ICE Models ...,"etcetera. So, and 200 km is quite a lot lah I ..."


In [3]:
# Save evaluation results to CSV for judging
export_path = 'eval_rag_results.csv'
eval_df.to_csv(export_path, index=False)
print(f'Saved: {export_path}')


Saved: eval_rag_results.csv
